# Práctica 1 Aprendizaje Automático: Predicción de Subscripción a un Producto Bancario

**Miembros del equipo:**  
Jose Luis Mejía Acuña  
Mireya Luque Perez

## Notebook 2: Carga del modelo final y predicciones sobre el conjunto de competición

Este segundo notebook tiene un objetivo mucho más concreto que el primero. Aquí no se repite el EDA ni la selección de modelo, porque esas decisiones ya se tomaron previamente.

En este cuaderno se va a:

1. cargar el modelo final ya entrenado,
2. cargar el fichero de competición,
3. aplicar el mismo tratamiento especial de la variable `pdays`,
4. generar las predicciones finales,
5. guardar el resultado en el fichero `predicciones.csv`.

De esta manera, el notebook queda centrado exclusivamente en la fase final de uso del modelo sobre datos nuevos, tal y como pide la práctica.

In [3]:
### IMPORTS PARA EL NOTEBOOK 2

import os
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## Carga del modelo final y localización de archivos

En este bloque se definen las rutas de trabajo y se cargan los dos elementos necesarios para esta parte final de la práctica:

- el fichero con el modelo final, `modelo_final.joblib`,
- el fichero de competición, `bank_competition.pkl`.

Además, se comprueba que ambos archivos existan realmente antes de continuar, para evitar errores posteriores difíciles de interpretar.

In [6]:
print("--- 1. CARGA DEL MODELO FINAL Y DEL DATASET DE COMPETICIÓN ---")

# ==========
# 1) Definimos rutas base
# ==========
ruta_modelo = Path("modelo_final.joblib")
ruta_competicion = Path("Datos/bank_competition.pkl")

# ==========
# 2) Comprobaciones de existencia
# ==========
if not ruta_modelo.exists():
    raise FileNotFoundError(f"No se encuentra el modelo final en la ruta: {ruta_modelo}")

if not ruta_competicion.exists():
    raise FileNotFoundError(f"No se encuentra el fichero de competición en la ruta: {ruta_competicion}")

# ==========
# 3) Carga del modelo y de los datos
# ==========
modelo_final = joblib.load(ruta_modelo)
df_competicion = pd.read_pickle(ruta_competicion)

print(f"Modelo cargado correctamente desde: {ruta_modelo}")
print(f"Dataset de competición cargado correctamente desde: {ruta_competicion}")
print(f"Número de instancias en competición: {df_competicion.shape[0]}")
print(f"Número de variables originales en competición: {df_competicion.shape[1]}")

display(df_competicion.head())

--- 1. CARGA DEL MODELO FINAL Y DEL DATASET DE COMPETICIÓN ---
Modelo cargado correctamente desde: modelo_final.joblib
Dataset de competición cargado correctamente desde: Datos/bank_competition.pkl
Número de instancias en competición: 162
Número de variables originales en competición: 16


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


La salida anterior permite comprobar que el modelo final está disponible y que el conjunto de competición se ha cargado correctamente.

También conviene observar las primeras filas del dataset para verificar que su estructura es coherente con la práctica y que, como era esperable, se trata de datos nuevos sobre los que se quiere predecir la variable `deposit`.

## Revisión de la estructura del conjunto de competición

Antes de generar predicciones, es importante revisar que el conjunto de competición tenga una estructura compatible con el modelo entrenado.

En particular, se va a comprobar:

- si la variable `deposit` aparece o no,
- si existe la columna `pdays`,
- y si las columnas coinciden con las esperadas por la pipeline final.

Esto es importante porque la práctica indica que el fichero `bank_competition_xx.pkl` no debe contener la variable objetivo, y porque en el primer notebook se decidió realizar un tratamiento específico de `pdays`.

In [7]:
print("--- 2. REVISIÓN DE LA ESTRUCTURA DEL DATASET DE COMPETICIÓN ---")


print("Columnas del dataset de competición:")
print(df_competicion.columns.tolist())

print("\n¿Está la variable objetivo 'deposit' en competición?")
print("deposit" in df_competicion.columns)

print("\n¿Está la variable 'pdays' en competición?")
print("pdays" in df_competicion.columns)

if hasattr(modelo_final, "feature_names_in_"):
    columnas_esperadas_modelo = list(modelo_final.feature_names_in_)
    print("\nNúmero de variables esperadas por el modelo final:", len(columnas_esperadas_modelo))
    print("Variables esperadas por el modelo:")
    print(columnas_esperadas_modelo)
else:
    columnas_esperadas_modelo = None
    print("\nAviso: el modelo cargado no expone 'feature_names_in_'.")

--- 2. REVISIÓN DE LA ESTRUCTURA DEL DATASET DE COMPETICIÓN ---
Columnas del dataset de competición:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']

¿Está la variable objetivo 'deposit' en competición?
False

¿Está la variable 'pdays' en competición?
True

Número de variables esperadas por el modelo final: 16
Variables esperadas por el modelo:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'previous', 'poutcome', 'contactado_antes']


En el primer notebook se decidió no utilizar `pdays` directamente como variable numérica estándar, porque el valor `-1` no representa una cantidad real de días, sino la ausencia de contacto previo o un valor desconocido.

Por tanto, en este segundo notebook hay que reproducir exactamente la misma transformación para mantener la consistencia entre entrenamiento y predicción:

- crear la variable binaria `contactado_antes`,
- eliminar la columna original `pdays`.

Este paso es imprescindible para que el modelo reciba entradas con la misma lógica que durante el entrenamiento.

In [8]:
print("--- 3. PREPROCESADO CONSISTENTE DE LA VARIABLE 'pdays' ---")

df_competicion_procesado = df_competicion.copy()

# Si por cualquier motivo aparece deposit, la eliminamos para evitar fugas o errores
if "deposit" in df_competicion_procesado.columns:
    print("Aviso: la columna 'deposit' aparece en competición y será eliminada.")
    df_competicion_procesado = df_competicion_procesado.drop(columns=["deposit"])

# Reproducimos el mismo tratamiento de pdays que en el notebook 1
if "pdays" in df_competicion_procesado.columns:
    df_competicion_procesado["contactado_antes"] = (df_competicion_procesado["pdays"] != -1).astype(int)
    df_competicion_procesado = df_competicion_procesado.drop(columns=["pdays"])
    
    print("Transformación aplicada correctamente:")
    print("- Se crea la variable binaria 'contactado_antes'")
    print("- Se elimina la columna original 'pdays'")
else:
    print("La columna 'pdays' no está presente, así que no ha sido necesario transformarla.")

print(f"\nNúmero de variables tras el preprocesado: {df_competicion_procesado.shape[1]}")
display(df_competicion_procesado.head())

--- 3. PREPROCESADO CONSISTENTE DE LA VARIABLE 'pdays' ---
Transformación aplicada correctamente:
- Se crea la variable binaria 'contactado_antes'
- Se elimina la columna original 'pdays'

Número de variables tras el preprocesado: 16


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,previous,poutcome,contactado_antes
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,1,other,1
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,0,unknown,0
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,0,unknown,0
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,0,unknown,0
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,3,success,1


## Alineación de columnas y generación de predicciones

Una vez aplicado el mismo tratamiento de `pdays`, el siguiente paso consiste en alinear el conjunto de competición con la estructura exacta que espera el modelo final.

Después, ya se pueden generar las predicciones y guardarlas en un fichero llamado `predicciones.csv`, que es uno de los entregables requeridos en la práctica.

In [9]:
print("--- 4. ALINEACIÓN DE COLUMNAS Y PREDICCIÓN FINAL ---")

# ==========
# 1) Alineación con las columnas esperadas por el modelo
# ==========
if columnas_esperadas_modelo is not None:
    faltan = [col for col in columnas_esperadas_modelo if col not in df_competicion_procesado.columns]
    sobran = [col for col in df_competicion_procesado.columns if col not in columnas_esperadas_modelo]

    if faltan:
        raise ValueError(f"Faltan columnas necesarias para el modelo: {faltan}")

    if sobran:
        print(f"Aviso: hay columnas extra en competición y no se usarán: {sobran}")

    X_competicion = df_competicion_procesado.reindex(columns=columnas_esperadas_modelo)
else:
    X_competicion = df_competicion_procesado.copy()

print(f"Número final de variables usadas para predecir: {X_competicion.shape[1]}")

# ==========
# 2) Predicciones
# ==========
predicciones_numericas = modelo_final.predict(X_competicion)
predicciones_etiquetas = pd.Series(predicciones_numericas).map({0: "no", 1: "yes"})

predicciones_df = pd.DataFrame({
    "deposit": predicciones_etiquetas
})

# ==========
# 3) Guardado
# ==========
predicciones_df.to_csv("predicciones.csv", index=False)

print("\nPredicciones generadas correctamente.")
print(f"Número de predicciones: {len(predicciones_df)}")
print("Archivo guardado correctamente: predicciones.csv")

print("\nDistribución de clases predichas:")
display(
    predicciones_df["deposit"]
    .value_counts()
    .rename_axis("Clase")
    .reset_index(name="Frecuencia")
)

print("\nPrimeras filas del archivo de salida:")
display(predicciones_df.head())

--- 4. ALINEACIÓN DE COLUMNAS Y PREDICCIÓN FINAL ---
Número final de variables usadas para predecir: 16

Predicciones generadas correctamente.
Número de predicciones: 162
Archivo guardado correctamente: predicciones.csv

Distribución de clases predichas:


,Clase,Frecuencia
0,no,93
1,yes,69



Primeras filas del archivo de salida:


,deposit
0,no
1,no
2,no
3,yes
4,yes


La celda anterior completa la tarea principal de este segundo notebook.

A partir del modelo final ya entrenado, se han generado las predicciones para el conjunto de competición y se han guardado en `predicciones.csv`. La tabla con la distribución de clases predichas sirve como comprobación rápida para detectar resultados extraños, por ejemplo si el modelo predijera la misma clase para todos los casos.

## Comprobación final del fichero generado

Por último, resulta útil verificar que el archivo `predicciones.csv` se ha creado correctamente y que tiene el formato esperado: una única columna llamada `deposit` con etiquetas categóricas `yes` o `no`.

In [10]:
print("--- 5. COMPROBACIÓN FINAL DE 'predicciones.csv' ---")

predicciones_guardadas = pd.read_csv("predicciones.csv")

print(f"Forma del archivo generado: {predicciones_guardadas.shape}")
print("Columnas del archivo:")
print(predicciones_guardadas.columns.tolist())

print("\nValores únicos en la columna 'deposit':")
print(predicciones_guardadas["deposit"].unique())

display(predicciones_guardadas.head())

--- 5. COMPROBACIÓN FINAL DE 'predicciones.csv' ---
Forma del archivo generado: (162, 1)
Columnas del archivo:
['deposit']

Valores únicos en la columna 'deposit':
<StringArray>
['no', 'yes']
Length: 2, dtype: str


,deposit
0,no
1,no
2,no
3,yes
4,yes


## Conclusión

Este segundo notebook queda dedicado exclusivamente al uso del modelo final sobre datos nuevos de competición. Su flujo es mucho más corto que el del primer notebook porque aquí ya no se comparan alternativas ni se ajustan hiperparámetros.

En resumen, en este cuaderno se ha:

- cargado el modelo final,
- cargado el conjunto de competición,
- reproducido el mismo tratamiento de `pdays`,
- generado el fichero `predicciones.csv`.

Con ello queda cubierta la parte del enunciado relativa al segundo notebook de predicción sobre datos de competición. Además ahora cubrimos nuestro caso de uso de Streamlit.

## 6. Comprobación de coincidencia entre Streamlit y la pipeline

Además de desplegar el modelo mediante `mystreamlit.py`, el enunciado pide demostrar que la aplicación funciona correctamente comparando sus predicciones con las obtenidas directamente por la pipeline en **dos nuevas instancias**.

En esta sección se construirán dos clientes de prueba con valores introducidos manualmente, se obtendrá la predicción de la pipeline cargada desde `modelo_final.joblib` y se dejará una tabla resumen lista para compararla con la app de Streamlit.

Estas dos instancias serán las que después se introducirán también en la aplicación web para comprobar que ambas predicciones coinciden exactamente.

In [11]:
print("--- 6.1 PREPARACIÓN DE DOS INSTANCIAS NUEVAS PARA LA COMPROBACIÓN ---")

# Aseguramos que el modelo esté cargado
if "modelo_final" not in globals():
    import joblib
    modelo_final = joblib.load("modelo_final.joblib")

columnas_modelo = list(modelo_final.feature_names_in_)
print("Columnas esperadas por el modelo:")
print(columnas_modelo)


def construir_instancia_desde_formulario(
    age,
    job,
    marital,
    education,
    default,
    balance,
    housing,
    loan,
    contact,
    month,
    duration,
    campaign,
    previous,
    poutcome,
    pdays,
    day=None,
    day_of_week=None
):
    """
    Construye una instancia con el mismo esquema lógico que usa la app.
    Si el modelo espera 'contactado_antes', se calcula a partir de pdays.
    Si el modelo espera 'pdays', se usa directamente.
    También gestiona de forma flexible 'day' o 'day_of_week'.
    """
    datos = {
        "age": int(age),
        "job": str(job),
        "marital": str(marital),
        "education": str(education),
        "default": str(default),
        "balance": float(balance),
        "housing": str(housing),
        "loan": str(loan),
        "contact": str(contact),
        "month": str(month),
        "duration": int(duration),
        "campaign": int(campaign),
        "previous": int(previous),
        "poutcome": str(poutcome),
    }

    if "contactado_antes" in columnas_modelo:
        datos["contactado_antes"] = int(int(pdays) != -1)

    if "pdays" in columnas_modelo:
        datos["pdays"] = int(pdays)

    if "day" in columnas_modelo:
        if day is None:
            raise ValueError("El modelo espera la columna 'day' y no se ha proporcionado.")
        datos["day"] = int(day)

    if "day_of_week" in columnas_modelo:
        if day_of_week is None:
            raise ValueError("El modelo espera la columna 'day_of_week' y no se ha proporcionado.")
        datos["day_of_week"] = str(day_of_week)

    X = pd.DataFrame([datos]).reindex(columns=columnas_modelo)

    columnas_con_nan = X.columns[X.isna().any()].tolist()
    if columnas_con_nan:
        raise ValueError(
            f"La instancia contiene NaN en estas columnas: {columnas_con_nan}. "
            "Revisa los campos introducidos."
        )

    return X


# ==========
# INSTANCIA NUEVA 1
# ==========
kwargs_1 = {
    "age": 39,
    "job": "unemployed",
    "marital": "divorced",
    "education": "unknown",
    "default": "yes",
    "balance": 549.50,
    "housing": "yes",
    "loan": "yes",
    "contact": "cellular",
    "month": "dec",
    "duration": 255,
    "campaign": 1,
    "previous": 2,
    "poutcome": "unknown",
    "pdays": 3,
}

# ==========
# INSTANCIA NUEVA 2
# ==========
kwargs_2 = {
    "age": 52,
    "job": "technician",
    "marital": "married",
    "education": "secondary",
    "default": "no",
    "balance": 1200.00,
    "housing": "no",
    "loan": "no",
    "contact": "telephone",
    "month": "jul",
    "duration": 180,
    "campaign": 2,
    "previous": 0,
    "poutcome": "unknown",
    "pdays": -1,
}

# Añadimos el campo correcto según lo que espere el modelo
if "day" in columnas_modelo:
    kwargs_1["day"] = 15
    kwargs_2["day"] = 7

if "day_of_week" in columnas_modelo:
    kwargs_1["day_of_week"] = "mon"
    kwargs_2["day_of_week"] = "wed"

X_cliente_1 = construir_instancia_desde_formulario(**kwargs_1)
X_cliente_2 = construir_instancia_desde_formulario(**kwargs_2)

print("Instancia 1 preparada correctamente.")
display(X_cliente_1)

print("Instancia 2 preparada correctamente.")
display(X_cliente_2)

--- 6.1 PREPARACIÓN DE DOS INSTANCIAS NUEVAS PARA LA COMPROBACIÓN ---
Columnas esperadas por el modelo:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'previous', 'poutcome', 'contactado_antes']
Instancia 1 preparada correctamente.


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,previous,poutcome,contactado_antes
0,39,unemployed,divorced,unknown,yes,549.5,yes,yes,cellular,15,dec,255,1,2,unknown,1


Instancia 2 preparada correctamente.


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,previous,poutcome,contactado_antes
0,52,technician,married,secondary,no,1200.0,no,no,telephone,7,jul,180,2,0,unknown,0


Las dos instancias anteriores representan dos clientes completamente nuevos, construidos manualmente para verificar el funcionamiento del despliegue.

Lo importante es que estos mismos valores serán los que se introducirán después en la app de Streamlit. De ese modo, la comparación entre pipeline y aplicación será exacta y podrá documentarse en el PDF con capturas.

In [12]:
print("--- 6.2 PREDICCIÓN DE LA PIPELINE PARA LAS DOS NUEVAS INSTANCIAS ---")

pred_1 = modelo_final.predict(X_cliente_1)[0]
pred_2 = modelo_final.predict(X_cliente_2)[0]

etiqueta_1 = "yes" if pred_1 == 1 else "no"
etiqueta_2 = "yes" if pred_2 == 1 else "no"

tabla_comprobacion = pd.DataFrame([
    {
        "Instancia": "Cliente 1",
        "Predicción pipeline": etiqueta_1,
        "age": kwargs_1["age"],
        "job": kwargs_1["job"],
        "marital": kwargs_1["marital"],
        "education": kwargs_1["education"],
        "default": kwargs_1["default"],
        "balance": kwargs_1["balance"],
        "housing": kwargs_1["housing"],
        "loan": kwargs_1["loan"],
        "contact": kwargs_1["contact"],
        "month": kwargs_1["month"],
        "duration": kwargs_1["duration"],
        "campaign": kwargs_1["campaign"],
        "previous": kwargs_1["previous"],
        "poutcome": kwargs_1["poutcome"],
        "pdays": kwargs_1["pdays"],
        "day/day_of_week": kwargs_1.get("day", kwargs_1.get("day_of_week")),
    },
    {
        "Instancia": "Cliente 2",
        "Predicción pipeline": etiqueta_2,
        "age": kwargs_2["age"],
        "job": kwargs_2["job"],
        "marital": kwargs_2["marital"],
        "education": kwargs_2["education"],
        "default": kwargs_2["default"],
        "balance": kwargs_2["balance"],
        "housing": kwargs_2["housing"],
        "loan": kwargs_2["loan"],
        "contact": kwargs_2["contact"],
        "month": kwargs_2["month"],
        "duration": kwargs_2["duration"],
        "campaign": kwargs_2["campaign"],
        "previous": kwargs_2["previous"],
        "poutcome": kwargs_2["poutcome"],
        "pdays": kwargs_2["pdays"],
        "day/day_of_week": kwargs_2.get("day", kwargs_2.get("day_of_week")),
    },
])

display(tabla_comprobacion)

tabla_comprobacion.to_csv("comprobacion_streamlit_vs_pipeline.csv", index=False)
print("Archivo guardado: comprobacion_streamlit_vs_pipeline.csv")

--- 6.2 PREDICCIÓN DE LA PIPELINE PARA LAS DOS NUEVAS INSTANCIAS ---


,Instancia,Predicción pipeline,age,job,marital,education,default,balance,housing,loan,contact,month,duration,campaign,previous,poutcome,pdays,day/day_of_week
0,Cliente 1,yes,39,unemployed,divorced,unknown,yes,549.5,yes,yes,cellular,dec,255,1,2,unknown,3,15
1,Cliente 2,no,52,technician,married,secondary,no,1200.0,no,no,telephone,jul,180,2,0,unknown,-1,7


Archivo guardado: comprobacion_streamlit_vs_pipeline.csv


La tabla anterior deja fijadas las dos instancias de prueba y la predicción obtenida directamente por la pipeline. A partir de aquí, solo falta introducir exactamente esos mismos valores en la aplicación de Streamlit y comprobar que el resultado mostrado por la app coincide con la columna **Predicción pipeline**.

El archivo `comprobacion_streamlit_vs_pipeline.csv` también se guarda como apoyo para la elaboración del PDF final de evidencias.

## 7. Qué se debe capturar para el PDF

Para completar la evidencia pedida en el enunciado, se recomienda incluir en el PDF:

1. Una captura de esta tabla del notebook donde aparezcan las dos instancias y sus predicciones de pipeline.
2. Una captura de la app de Streamlit para el **Cliente 1**, mostrando los valores introducidos y la predicción obtenida.
3. Una captura de la app de Streamlit para el **Cliente 2**, mostrando los valores introducidos y la predicción obtenida.
4. Una frase final indicando que, en ambos casos, la predicción de Streamlit coincide con la predicción producida por la pipeline.

Con ello queda demostrada la correcta correspondencia entre el modelo desplegado y el modelo guardado en `modelo_final.joblib`.